# Aileron Design — Cruise-Phase Roll Authority
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

`control_vane_design` (NB3) sizes jet-vane control authority from **hover**
thrust only. Because `q_jet = T / A_disk` scales linearly with thrust
(actuator-disk momentum theory), jet-vane authority collapses in cruise,
where thrust is only a small fraction of hover thrust (set by cruise L/D).
This notebook sizes wing **ailerons** as the primary roll actuator in
wing-borne cruise -- ailerons use wing dynamic pressure instead, which
does not depend on EDF throttle setting.

Jet vanes remain primary for all three axes in hover/transition (where
aero surfaces are ineffective) and stay available as a roll backup in
cruise -- nothing is removed, this notebook only adds a second roll path
where the first one is weakest.

---

## Inputs

- `out/airfoil.yaml` *(from Notebook 2 — wing airfoil, t/c, Oswald e)*
- `out/control_vanes.yaml` *(from Notebook 3 — jet-vane authority, hover
  q_jet, slender-body roll/pitch inertia estimates)*
- `config/aileron.yaml` *(aileron geometry, deflection limits, hardware
  mass)*
- `config/aerodynamics.yaml` *(shared `ddot_min_deg_s2` requirement)*

## Outputs

- `out/aileron.yaml`
- `out/aileron_authority_curves.png`

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design.aileron_design import (
    AileronParams, size_aileron, vane_cruise_authority, write_aileron_yaml,
)
from conceptual_design import reports
from conceptual_design.plots import plot_aileron_authority

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` -- same pattern as NB2/NB3, so this
notebook stays consistent with the upstream design state -- and load the
wing/vane handoffs written by the previous notebooks.

---

In [ ]:
# -- Re-run the sizing loop from config/ + upstream handoffs -------------------
inp, result = solve_design_point(CONFIG_PATH)
ail_p = AileronParams.from_yaml(CONFIG_PATH / "aileron.yaml")

af    = load_handoff(OUT_PATH, "airfoil")
vanes = load_handoff(OUT_PATH, "control_vanes")

MTOW_kg  = result.m_total_kg
W_N      = MTOW_kg * inp.env.g
ddot_min = inp.aero.ddot_min_deg_s2

print(f"Converged MTOW : {MTOW_kg:.3f} kg  ({W_N:.2f} N)")
print(f"Wing            : b = {result.wing.b_wing*1e3:.0f} mm, "
      f"MAC = {result.wing.chord_mean*1e3:.1f} mm")
print(f"V_cruise        : {inp.mission.V_cruise:.1f} m/s")
print(f"ddot_min        : {ddot_min:.0f} deg/s^2  (config/aerodynamics.yaml, shared with NB3)")

# Section 2 — Why Jet Vanes Alone Aren't Enough in Cruise

Actuator-disk momentum theory gives `q_jet = T / A_disk` -- jet-vane force
and moment are exactly proportional to EDF thrust. NB3 sizes vane
authority at **hover** thrust (`T_hover = s_ratio * W`); in wing-borne
cruise the EDF only needs to overcome drag (`T_cruise = W / (L/D)_cruise`),
a small fraction of hover thrust. Scaling NB3's hover `ddot_roll` by that
same thrust ratio gives the jet-vane-alone authority actually available
during the ~900 s cruise leg.

---

In [ ]:
# -- Residual jet-vane authority in cruise (q_jet scales with thrust) ----------
auth = vane_cruise_authority(W_N, inp.prop.s_ratio, af["LD_cruise"], vanes)
reports.print_vane_cruise_authority(auth, ddot_min)

# Section 3 — Aileron Geometry and Cruise Roll Authority

One aileron occupies the outboard `span_frac_wing` fraction of the wing
half-span, with chord `chord_frac` of the wing MAC. Flap effectiveness
comes from Glauert thin-airfoil theory; the wing's own lift-curve slope
(2D + finite-span correction) is reused from `airfoil_selection.py`
rather than re-derived. Roll inertia is the same slender-body estimate
NB3 already computed (`I_roll_kgm2`), read directly from
`out/control_vanes.yaml`.

---

In [ ]:
ail = size_aileron(
    b_wing_m=result.wing.b_wing, chord_mean_m=result.wing.chord_mean,
    tc_ratio=af["tc_ratio"], AR=inp.aero.AR, e_oswald=af["e_oswald"],
    V_cruise=inp.mission.V_cruise, rho=inp.env.rho,
    I_roll=vanes["I_roll_kgm2"], p=ail_p,
)

ddot_roll_total_cruise = ail.ddot_roll_deg_s2 + auth.ddot_roll_cruise
cruise_ok = ddot_roll_total_cruise >= ddot_min

print(f"Aileron chord         : {ail.c_aileron_m*1e3:.1f} mm  "
      f"(chord_frac={ail_p.chord_frac:.2f})")
print(f"Aileron span          : {ail.b_aileron_m*1e3:.1f} mm  "
      f"(span_frac_wing={ail_p.span_frac_wing:.2f})")
print(f"Aileron area (1 side) : {ail.S_aileron_m2*1e4:.2f} cm^2")
print(f"Spanwise arm          : {ail.y_arm_m*1e3:.1f} mm")
print(f"Flap effectiveness tau: {ail.tau:.3f}")
print(f"Cl_delta              : {ail.Cl_delta_per_rad:.3f} /rad")
print(f"q_cruise              : {ail.q_cruise_Pa:.2f} Pa")
print()
print(f"Aileron-alone ddot_roll (cruise) : {ail.ddot_roll_deg_s2:.1f} deg/s^2")
print(f"Vane-alone ddot_roll (cruise)    : {auth.ddot_roll_cruise:.1f} deg/s^2")
print(f"Combined ddot_roll (cruise)      : {ddot_roll_total_cruise:.1f} deg/s^2")
print(f"Requirement                      : {ddot_min:.0f} deg/s^2")
print(f"Cruise roll authority             : {'OK' if cruise_ok else 'FAIL -- re-size aileron'}")
print()
print(f"Hinge moment @ delta_max : {ail.M_hinge_max_Nmm:.3f} N*mm")
print(f"Required servo torque    : >= {ail.servo_torque_req_gcm:.1f} g*cm  "
      f"(SF={ail_p.SF_servo}, +20% margin)")

# Section 4 — Deflection Sweep and Authority Curves

Same style as NB3's `vane_authority_curves.png`: force/moment vs.
deflection, and the combined (aileron + residual vane) cruise roll
authority against the shared `ddot_min_deg_s2` floor.

---

In [ ]:
plot_aileron_authority(ail, ail_p, vanes["I_roll_kgm2"],
                       auth.ddot_roll_cruise, ddot_min,
                       OUT_PATH / "aileron_authority_curves.png")

# Section 5 — Output Export

`out/aileron.yaml` -- consumed by `fuselage_design.py` (servo/linkage mass
carve-out) and kept for reference by downstream notebooks.

---

In [ ]:
write_aileron_yaml(
    ail, ail_p, OUT_PATH / "aileron.yaml",
    ddot_roll_vane_cruise_deg_s2=auth.ddot_roll_cruise,
    ddot_min_deg_s2=ddot_min,
)
print(f"Aileron design written -> {OUT_PATH / 'aileron.yaml'}")

# Section 6 — Design Summary

---

In [ ]:
reports.print_aileron_summary(ail, ail_p, auth, ddot_roll_total_cruise,
                              ddot_min, cruise_ok)